# 🔍 Hybrid Retrieval: FAISS + BM25 + RRF
**Mata Kuliah:** Natural Language Processing (UAS)

Notebook ini membangun sistem retrieval hybrid untuk dataset hukum Klinik Hukumonline.
* **Dense Retrieval:** FAISS `IndexFlatIP` dengan vektor `multilingual-e5-large` (1024-dim, L2-normalized)
* **Sparse Retrieval:** BM25 berbasis teks `title + question + summarize`
* **Fusion:** Reciprocal Rank Fusion (RRF, k=60)
* **Input:** `vectors.npy` dan `documents.json` dari `embedding.ipynb`

## 1. Install Dependensi

In [ ]:
!pip install -q faiss-cpu rank-bm25 sentence-transformers torch numpy

## 2. Import & Muat Data

In [ ]:
import os
import json
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

# Auto-detect lokasi file (lokal atau Colab)
VECTORS_PATH = "vectors.npy"
DOCS_PATH    = "documents.json"

assert os.path.exists(VECTORS_PATH), f"File tidak ditemukan: {VECTORS_PATH}"
assert os.path.exists(DOCS_PATH),    f"File tidak ditemukan: {DOCS_PATH}"

vectors = np.load(VECTORS_PATH).astype(np.float32)
with open(DOCS_PATH, "r", encoding="utf-8") as f:
    docs = json.load(f)

print(f"Vectors shape : {vectors.shape}")   # (7172, 1024)
print(f"Docs count    : {len(docs)}")        # 7172
print(f"Norm sample   : {np.linalg.norm(vectors[0]):.4f}")  # ~1.0

## 3. Muat Model E5 (untuk Encode Query)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")

model = SentenceTransformer("intfloat/multilingual-e5-large", device=device)
print("Model berhasil dimuat!")

## 4. Build FAISS Index (Dense Retrieval)
`IndexFlatIP` = Inner Product (sama dengan cosine similarity karena vektor sudah L2-normalized)

In [ ]:
dim = vectors.shape[1]  # 1024
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(vectors)

print(f"FAISS index built: {faiss_index.ntotal} vectors, dim={dim}")

## 5. Build BM25 Index (Sparse Retrieval)
Teks yang diindeks: `title + question + summarize` (konsisten dengan field yang di-embed)

In [ ]:
# Tokenisasi sederhana: lowercase + split whitespace
bm25_corpus = [
    (doc["title"] + " " + doc["question"] + " " + doc["summarize"]).lower().split()
    for doc in docs
]

bm25_index = BM25Okapi(bm25_corpus)
print(f"BM25 index built: {len(bm25_corpus)} dokumen")

## 6. Fungsi Hybrid Search (FAISS + BM25 + RRF)
**Reciprocal Rank Fusion (RRF):** `score = 1/(k + rank_dense) + 1/(k + rank_sparse)` dengan `k=60`

In [ ]:
def hybrid_search(query: str, top_k: int = 5, rrf_k: int = 60) -> list[dict]:
    """
    Hybrid retrieval: FAISS (dense) + BM25 (sparse) fused dengan RRF.
    
    Args:
        query   : Teks pertanyaan dari user
        top_k   : Jumlah dokumen yang dikembalikan
        rrf_k   : Konstanta RRF (default=60, standar dari paper Cormack et al.)
    
    Returns:
        List of dict: dokumen terurut berdasarkan skor RRF (descending)
    """
    # --- Dense retrieval (FAISS) ---
    q_vec = model.encode(
        f"query: {query.strip()}",
        normalize_embeddings=True,
        convert_to_numpy=True
    ).reshape(1, -1).astype(np.float32)

    candidate_pool = min(top_k * 10, len(docs))  # ambil lebih banyak kandidat untuk di-fuse
    dense_scores, dense_ids = faiss_index.search(q_vec, candidate_pool)
    dense_ids = dense_ids[0].tolist()           # list of doc indices, ranked by score

    # --- Sparse retrieval (BM25) ---
    bm25_scores = bm25_index.get_scores(query.lower().split())
    sparse_ids  = np.argsort(bm25_scores)[::-1][:candidate_pool].tolist()

    # --- RRF Fusion ---
    rrf_scores: dict[int, float] = {}

    for rank, doc_id in enumerate(dense_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank + 1)

    for rank, doc_id in enumerate(sparse_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank + 1)

    # Urutkan berdasarkan skor RRF (descending), ambil top_k
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for doc_id, rrf_score in ranked:
        doc = docs[doc_id].copy()
        doc["rrf_score"] = round(rrf_score, 6)
        doc["dense_score"] = round(float(np.dot(vectors[doc_id], q_vec[0])), 4)
        results.append(doc)

    return results


print("Fungsi hybrid_search() siap digunakan.")

## 7. Uji Coba Retrieval

In [ ]:
def print_results(query: str, top_k: int = 3):
    results = hybrid_search(query, top_k=top_k)
    print(f"=== Query: '{query}' ===\n")
    for i, doc in enumerate(results, 1):
        print(f"[{i}] RRF={doc['rrf_score']:.5f}  Dense={doc['dense_score']:.4f}")
        print(f"     Judul: {doc['title']}")
        print(f"     Kasus: {doc['question'][:120]}...")
        print(f"     URL  : {doc['source']}\n")


# --- Test queries ---
print_results("bagaimana cara membeli tanah girik tanpa sertifikat yang aman?")

In [ ]:
print_results("bos gamau cairin uang lembur lapor kemana ya?")

In [ ]:
print_results("duit kantor kepake sama temen tapi udah dicicil balikin apa tetep dipenjara?")

In [ ]:
print_results("how can a foreigner legally own property or land in bali?")